# Provider-Aware SISA

## 1. Purpose and provider-aware assumption

This is an exploratory SISA experiment. Standard hash-based SISA remains the request-agnostic reference; Provider-Aware SISA asks whether plausible operational information available before deletion can make later requests more local.

The assignment uses hospital provenance and donor-linked relationships, not exact future deletion-policy membership. Recipient histories remain intact, linked donor histories form one component, components are allocated across five provider-aware shards, and chronological slices represent data arrival. The assignment is frozen before the established deletion memberships are loaded.

Policy-Aware SISA is different: it deliberately uses exact predefined policy membership as an optimistic sensitivity analysis. Provider-Aware SISA makes the more realistic assumption that provider and relational information may be known in advance, while future forget sets are not.

The experiment tests structural localisation, retained utility, forgetting similarity and runtime as separate outcomes. Results are synthetic and configuration-specific, and do not establish privacy or clinical validity.

## 2. Load frozen data and references

The notebook uses five shards, five slices, seed 42, threshold 0.62 and the established `24 → 64 → 32 → 1` PyTorch model. Timing uses the same sequential `perf_counter` boundary as the other method notebooks, and no network library is imported.

The standard SISA, Policy-Aware SISA and main-method artifacts are read-only references. Provider-aware models and results remain isolated in their own directories.

In [1]:
# Import only the libraries required by the established notebook workflow.
import copy
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy.stats import ks_2samp


In [2]:
# Import only the libraries required by the established notebook workflow.
from sklearn.metrics import (accuracy_score, average_precision_score,
    balanced_accuracy_score, confusion_matrix, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [3]:
# Keep paths portable within the clean submission workspace.
def locate_submission_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, cwd / "final_submission"]:
        direct = candidate / "data/final/kidney_transplant_assessments.csv"
        nested = candidate / "final_submission/data/final/kidney_transplant_assessments.csv"
        if direct.is_file(): return candidate
        if nested.is_file(): return candidate / "final_submission"
    raise FileNotFoundError("Could not locate final_submission")

ROOT = locate_submission_root()
DATA_PATH = ROOT / "data/final/kidney_transplant_assessments.csv"
IDENTITY_PATH = ROOT / "data/final/kidney_transplant_identity.csv"
FEATURE_PATH = ROOT / "data/final/classifier_feature_list.json"
SPLIT_PATH = ROOT / "processed_data/split_assignments.csv"
MEMBERSHIP_PATH = ROOT / "processed_data/deletion_scenario_membership.csv"
MODEL_DIR = ROOT / "models/sisa/provider_aware"


In [4]:
ORIGINAL_DIR, UPDATED_DIR = MODEL_DIR / "original", MODEL_DIR / "unlearned"
RESULT_DIR = ROOT / "results/sisa/provider_aware"

In [5]:
# Isolate provider-aware outputs from every frozen comparison method.
BASELINE_DIR = ROOT / "models/baseline"
FULL_RESULT_DIR = ROOT / "results/full_retraining"
STANDARD_RESULT_DIR = ROOT / "results/sisa/hash_based"
POLICY_RESULT_DIR = ROOT / "results/sisa/policy_aware"
STANDARD_MODEL_DIR = ROOT / "models/sisa/hash_based"
POLICY_MODEL_DIR = ROOT / "models/sisa/policy_aware"
PROVIDER_NOTEBOOK = ROOT / "notebooks/05_provider_aware_sisa.ipynb"

excluded_prefixes = ("models/sisa/provider_aware/", "results/sisa/provider_aware/")
protected_paths = []
for path in sorted(p for p in ROOT.rglob("*") if p.is_file()):
    relative = path.relative_to(ROOT).as_posix()
    if path == PROVIDER_NOTEBOOK or relative.startswith(excluded_prefixes): continue
    protected_paths.append(path)

def sha256_file(path): return hashlib.sha256(path.read_bytes()).hexdigest()
protected_hashes_before = {p.relative_to(ROOT).as_posix(): sha256_file(p) for p in protected_paths}


In [6]:
for path in [ORIGINAL_DIR, UPDATED_DIR, RESULT_DIR]: path.mkdir(parents=True, exist_ok=True)

In [7]:
# Match the standard SISA settings so partitioning remains the main difference.
SEED, N_SHARDS, N_SLICES = 42, 5, 5
BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY = 512, 0.001, 0.0001
MAX_EPOCHS_PER_SLICE, EARLY_STOPPING_PATIENCE = 12, 3
THRESHOLD, EPSILON = 0.62, 1e-12
SCENARIOS = ["recipient_withdrawal", "donor_withdrawal", "hospital_removal",
             "invalid_consent", "retention_expiry"]

def derived_seed(*parts):
    text = ":".join(map(str, (SEED, *parts)))
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

reset_seed()
torch.use_deterministic_algorithms(True)


In [8]:
# Keep shard and slice operations explicit so SISA provenance remains auditable.
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
display(pd.DataFrame([{"seed":SEED,"device":str(DEVICE),"shards":N_SHARDS,
                       "slices_per_shard":N_SLICES,"threshold":THRESHOLD}]))

,seed,device,shards,slices_per_shard,threshold
0,42,cpu,5,5,0.62


The dataset, identity table, permanent splits and frozen feature contract are loaded and checked here. Deletion membership is deliberately not read yet. Targets and clinical features are available for later training but cannot influence assignment.

In [9]:
# Load the frozen data, split and feature contract without modification.
assessments = pd.read_csv(DATA_PATH, parse_dates=["assessment_date", "retention_expiry_date"])
identity = pd.read_csv(IDENTITY_PATH)
splits = pd.read_csv(SPLIT_PATH, usecols=["recipient_id", "donor_id", "assessment_count", "split"])
feature_contract = json.loads(FEATURE_PATH.read_text())
features, target = feature_contract["classifier_features"], feature_contract["target"]
baseline_config = json.loads((BASELINE_DIR / "model_configuration.json").read_text())
preprocessor = joblib.load(BASELINE_DIR / "baseline_preprocessor.joblib")

reference_files = [FULL_RESULT_DIR / "run_summary.json",
    STANDARD_RESULT_DIR / "run_summary.json", POLICY_RESULT_DIR / "run_summary.json",
    STANDARD_RESULT_DIR / "utility_metrics.csv", POLICY_RESULT_DIR / "utility_metrics.csv"]
assert all(path.is_file() for path in reference_files)
assert len(features) == 18 and baseline_config["input_dim_processed"] == 24
assert baseline_config["threshold"] == THRESHOLD
deletion_masks_loaded = False

In [10]:
# Confirm recipient histories have one donor and one recorded provider.
split_map = splits.set_index("recipient_id")["split"]
assessments["original_split"] = assessments["recipient_id"].map(split_map)
recipient_static = assessments.groupby("recipient_id").agg(
    donor_count=("donor_id","nunique"), hospital_count=("hospital_id","nunique"),
    assessment_count=("assessment_id","size"), donor_id=("donor_id","first"))
checks = {
    "unique assessment IDs": assessments["assessment_id"].is_unique,
    "every assessment has split": assessments["original_split"].notna().all(),
    "six rows per recipient": recipient_static["assessment_count"].eq(6).all(),
    "one donor per recipient": recipient_static["donor_count"].eq(1).all(),
    "recipient split alignment": set(splits.recipient_id) == set(assessments.recipient_id),
    "donor split boundary": splits.groupby("donor_id")["split"].nunique().max() == 1,
    "recipient identities exist": set(assessments.recipient_id).issubset(set(identity.entity_id)),
    "donor identities exist": set(assessments.donor_id).issubset(set(identity.entity_id)),
}


In [11]:
input_validation = pd.DataFrame([{"check":k,"pass":bool(v)} for k,v in checks.items()])
display(input_validation.assign(status=input_validation["pass"].map({True:"PASS",False:"FAIL"})))
assert input_validation["pass"].all() and not deletion_masks_loaded

,check,pass,status
0,unique assessment IDs,True,PASS
1,every assessment has split,True,PASS
2,six rows per recipient,True,PASS
3,one donor per recipient,True,PASS
4,recipient split alignment,True,PASS
5,donor split boundary,True,PASS
6,recipient identities exist,True,PASS
7,donor identities exist,True,PASS


The input checks pass without consulting future deletion membership.

## 3. Build the provider-aware assignment

Hospitals are treated as plausible data providers that may later end a data-sharing agreement. Each recipient's six assessments remain together, and recipients sharing a donor form one atomic component. A component's home hospital is the hospital contributing the most rows, with hospital ID providing a deterministic tie-break.

Donor linkage takes priority over perfect hospital locality. The small table below illustrates the planned two-hospital-per-shard layout while the accompanying guard confirms deletion masks are still unavailable.

In [12]:
# Illustrate the intended provider pairing before constructing it.
example_layout = pd.DataFrame({"shard":[f"Shard {i}" for i in range(1,6)],
                               "illustration":["two hospitals"]*5})
display(example_layout)
assert not deletion_masks_loaded

,shard,illustration
0,Shard 1,two hospitals
1,Shard 2,two hospitals
2,Shard 3,two hospitals
3,Shard 4,two hospitals
4,Shard 5,two hospitals


Linked recipient–donor components are constructed only from recipient IDs, donor links, hospital provenance, row counts and dates. Targets, classifier features and future deletion membership are excluded.

In [13]:
# Use provider information available before any deletion request.
train_rows = assessments.loc[assessments["original_split"].eq("train")]
allowed_assignment_columns = ["recipient_id","donor_id","hospital_id","assessment_id","assessment_date"]
assignment_source = train_rows[allowed_assignment_columns].copy()
forbidden_assignment_columns = [target, *features, "training_consent_status",
    "training_consent_version", "retention_expiry_date", "previous_rejection"]
assert set(assignment_source.columns).isdisjoint(forbidden_assignment_columns)

recipient_rows = assignment_source.groupby(["recipient_id","donor_id","hospital_id"],as_index=False).agg(
    assessment_rows=("assessment_id","size"), anchor_date=("assessment_date","min"))
component_hospitals = recipient_rows.groupby(["donor_id","hospital_id"],as_index=False).agg(
    rows=("assessment_rows","sum"), recipients=("recipient_id","nunique"))
component_hospitals = component_hospitals.sort_values(["donor_id","rows","hospital_id"],
                                                       ascending=[True,False,True])
home_hospital = component_hospitals.drop_duplicates("donor_id").set_index("donor_id")["hospital_id"]

In [14]:
# Preserve donor-linked recipient histories as one assignment component.
components = recipient_rows.groupby("donor_id",as_index=False).agg(
    recipient_count=("recipient_id","nunique"), row_count=("assessment_rows","sum"),
    hospital_count=("hospital_id","nunique"), anchor_date=("anchor_date","min"))
components = components.rename(columns={"donor_id":"component_id"})
components["home_hospital_id"] = components["component_id"].map(home_hospital)
component_summary = pd.DataFrame([{
    "components":len(components),
    "single_recipient_components":int(components.recipient_count.eq(1).sum()),
    "shared_donor_components":int(components.recipient_count.gt(1).sum()),
    "cross_hospital_components":int(components.hospital_count.gt(1).sum()),
    "largest_component_recipients":int(components.recipient_count.max()),
}])
components.to_csv(RESULT_DIR / "provider_components.csv",index=False)
display(component_summary)
assert components.row_count.sum() == len(train_rows) and not deletion_masks_loaded

,components,single_recipient_components,shared_donor_components,cross_hospital_components,largest_component_recipients
0,5601,4198,1403,1218,2


Hospitals are ranked by the total weight of their atomic components. The largest provider group is paired with the smallest, then the second largest with the second smallest, producing five shards before deletion masks are loaded.

In [15]:
# Pair large and small home-provider groups to limit shard imbalance.
hospital_weights = components.groupby("home_hospital_id",as_index=False)["row_count"].sum()
hospital_weights = hospital_weights.sort_values(["row_count","home_hospital_id"],
                                                 ascending=[False,True]).reset_index(drop=True)
assert len(hospital_weights) == 10
pair_rows = []
for index in range(N_SHARDS):
    large = hospital_weights.iloc[index]
    small = hospital_weights.iloc[-(index + 1)]
    for role, row in [("larger",large),("smaller",small)]:
        pair_rows.append({"shard_id":index+1,"pair_role":role,
            "hospital_id":row.home_hospital_id,"component_rows":int(row.row_count)})
hospital_pairing = pd.DataFrame(pair_rows)
assert hospital_pairing.hospital_id.is_unique and len(hospital_pairing)==10
hospital_pairing.to_csv(RESULT_DIR / "hospital_pairing.csv",index=False)


In [16]:
display(hospital_pairing.groupby("shard_id").agg(
    hospitals=("hospital_id",lambda x:" | ".join(x)), rows=("component_rows","sum")).reset_index())

,shard_id,hospitals,rows
0,1,V32P-H01 | V32P-H10,10908
1,2,V32P-H02 | V32P-H09,8808
2,3,V32P-H03 | V32P-H08,7848
3,4,V32P-H04 | V32P-H07,7536
4,5,V32P-H05 | V32P-H06,6924


In [17]:
# Assign whole donor components even when linked histories span providers.
hospital_to_shard = hospital_pairing.set_index("hospital_id")["shard_id"]
components["shard_id"] = components["home_hospital_id"].map(hospital_to_shard).astype(int)
recipient_assignment = recipient_rows.rename(columns={"donor_id":"component_id"}).merge(
    components[["component_id","home_hospital_id","shard_id"]],on="component_id",validate="many_to_one")
provider_shard_assignment = recipient_assignment[["recipient_id","component_id","hospital_id",
    "home_hospital_id","shard_id"]].drop_duplicates("recipient_id").sort_values("recipient_id")
provider_shard_assignment.to_csv(RESULT_DIR / "provider_shard_assignment.csv",index=False)
shard_assignment_hash = sha256_file(RESULT_DIR / "provider_shard_assignment.csv")
assert provider_shard_assignment.recipient_id.is_unique and not deletion_masks_loaded

The assignment checks require complete and unique training-row coverage, one shard per recipient and donor component, and five non-empty shards. Hospital fragmentation is reported as a diagnostic rather than corrected after the result is known.

In [18]:
# Quantify provider spillover created by preserving relational groups.
train_frame = train_rows.merge(provider_shard_assignment,on=["recipient_id","hospital_id"],
                               how="left",validate="many_to_one")
shard_summary = train_frame.groupby("shard_id",as_index=False).agg(
    training_rows=("assessment_id","size"), recipients=("recipient_id","nunique"),
    donor_components=("component_id","nunique"), hospitals_present=("hospital_id","nunique"))
mean_rows = shard_summary.training_rows.mean()
shard_summary["balance_deviation_pct"] = 100*(shard_summary.training_rows-mean_rows)/mean_rows
fragmentation = train_frame.groupby("hospital_id",as_index=False).agg(
    shards_present=("shard_id","nunique"), rows=("assessment_id","size"))
fragmentation["fragmented"] = fragmentation.shards_present.gt(1)
train_frame["assigned_hospital_shard"] = train_frame.hospital_id.map(hospital_to_shard)
train_frame["hospital_spillover"] = train_frame.shard_id.ne(train_frame.assigned_hospital_shard)
cross_hospital_spillover_rows = int(train_frame.hospital_spillover.sum())
display(shard_summary)
display(fragmentation)

,shard_id,training_rows,recipients,donor_components,hospitals_present,balance_deviation_pct
0,1,10908,1818,1358,10,29.782981
1,2,8808,1468,1149,9,4.797259
2,3,7848,1308,1061,8,-6.624786
3,4,7536,1256,1051,7,-10.336950
4,5,6924,1154,982,6,-17.618504


,hospital_id,shards_present,rows,fragmented
0,V32P-H01,1,7620,False
1,V32P-H02,2,6306,True
2,V32P-H03,3,5424,True
3,V32P-H04,4,4938,True
4,V32P-H05,5,4314,True
5,V32P-H06,5,3678,True
6,V32P-H07,5,3540,True
7,V32P-H08,5,2868,True
8,V32P-H09,5,2070,True
9,V32P-H10,5,1266,True


In [19]:
# Confirm every recipient and donor component remains in one shard.
shard_checks = {
    "every training row once": len(train_frame)==len(train_rows) and train_frame.assessment_id.is_unique,
    "every recipient one shard": train_frame.groupby("recipient_id").shard_id.nunique().max()==1,
    "every donor component one shard": train_frame.groupby("component_id").shard_id.nunique().max()==1,
    "no missing assignments": train_frame[["shard_id","component_id"]].notna().all().all(),
    "five non-empty shards": set(train_frame.shard_id)==set(range(1,6)),
    "no deletion masks loaded": not deletion_masks_loaded,
}
shard_validation = pd.DataFrame([{"check":k,"pass":bool(v)} for k,v in shard_checks.items()])
shard_summary.to_csv(RESULT_DIR / "shard_summary.csv",index=False)
fragmentation.to_csv(RESULT_DIR / "hospital_fragmentation.csv",index=False)
shard_validation.to_csv(RESULT_DIR / "shard_validation.csv",index=False)
assert shard_validation["pass"].all()
display(shard_validation.assign(status=shard_validation["pass"].map({True:"PASS",False:"FAIL"})))

,check,pass,status
0,every training row once,True,PASS
1,every recipient one shard,True,PASS
2,every donor component one shard,True,PASS
3,no missing assignments,True,PASS
4,five non-empty shards,True,PASS
5,no deletion masks loaded,True,PASS


Donor-linked histories can span hospitals, so preserving relational groups creates provider spillover. This is kept and quantified because it may weaken the expected localisation advantage.

## 4. Create shards and chronological slices

Within each shard, components are ordered by their earliest assessment date and stable component ID. Weighted midpoint allocation creates five approximately balanced chronological slices without using targets or deletion-request information.

In [20]:
# Order components chronologically before future deletion masks are loaded.
slice_rows = []
for shard_id in range(1,N_SHARDS+1):
    ordered = components.loc[components.shard_id.eq(shard_id)].sort_values(
        ["anchor_date","component_id"]).copy()
    total = ordered.row_count.sum()
    midpoint = ordered.row_count.cumsum()-ordered.row_count/2
    ordered["slice_id"] = np.floor(midpoint/total*N_SLICES).astype(int).clip(0,N_SLICES-1)+1
    slice_rows.append(ordered[["component_id","anchor_date","shard_id","slice_id","row_count"]])
component_slices = pd.concat(slice_rows,ignore_index=True)
provider_assignment = provider_shard_assignment.merge(
    component_slices[["component_id","anchor_date","slice_id"]],on="component_id",validate="many_to_one")
provider_assignment = provider_assignment.sort_values("recipient_id").reset_index(drop=True)
provider_assignment.to_csv(RESULT_DIR / "provider_sisa_assignment.csv",index=False)
slice_assignment_hash = sha256_file(RESULT_DIR / "provider_sisa_assignment.csv")
train_frame = train_rows.merge(provider_assignment,on=["recipient_id","hospital_id"],validate="many_to_one")

Slice checks require five non-empty slices per shard, complete row coverage, chronological ordering, and one slice per recipient and donor component. These assertions stay beside the assignment they protect.

In [21]:
# Confirm every component occupies one ordered slice.
slice_summary = train_frame.groupby(["shard_id","slice_id"],as_index=False).agg(
    recipients=("recipient_id","nunique"), rows=("assessment_id","size"),
    donor_components=("component_id","nunique"),
    minimum_date=("assessment_date","min"), maximum_date=("assessment_date","max"))
slice_summary["mean_shard_slice_rows"] = slice_summary.groupby("shard_id")["rows"].transform("mean")
slice_summary["balance_deviation_pct"] = 100*(slice_summary.rows-slice_summary.mean_shard_slice_rows)/slice_summary.mean_shard_slice_rows
date_order_pass = all(group.sort_values("slice_id").minimum_date.is_monotonic_increasing
                      for _,group in slice_summary.groupby("shard_id"))


In [22]:
# Keep shard and slice operations explicit so SISA provenance remains auditable.
slice_checks = {
    "25 non-empty shard-slices":len(slice_summary)==25 and (slice_summary.rows>0).all(),
    "monotonic slice dates":date_order_pass,
    "recipient one slice":train_frame.groupby("recipient_id").slice_id.nunique().max()==1,
    "donor component one slice":train_frame.groupby("component_id").slice_id.nunique().max()==1,
    "complete unique rows":len(train_frame)==len(train_rows) and train_frame.assessment_id.is_unique,
    "no deletion masks loaded":not deletion_masks_loaded,
}
slice_validation=pd.DataFrame([{"check":k,"pass":bool(v)} for k,v in slice_checks.items()])
slice_summary.to_csv(RESULT_DIR / "slice_summary.csv",index=False)
slice_validation.to_csv(RESULT_DIR / "slice_validation.csv",index=False)
display(slice_summary)
assert slice_validation["pass"].all()

,shard_id,slice_id,recipients,rows,donor_components,minimum_date,maximum_date,mean_shard_slice_rows,balance_deviation_pct
0,1,1,364,2184,245,2023-11-10,2025-06-16,2181.6,0.110011
1,1,2,363,2178,247,2023-11-30,2025-06-29,2181.6,-0.165017
2,1,3,364,2184,264,2023-12-21,2025-06-27,2181.6,0.110011
3,1,4,363,2178,287,2024-03-10,2025-06-26,2181.6,-0.165017
4,1,5,364,2184,315,2024-07-14,2025-06-29,2181.6,0.110011
5,2,1,294,1764,199,2023-11-10,2025-06-14,1761.6,0.136240
6,2,2,293,1758,219,2023-12-01,2025-06-20,1761.6,-0.204360
7,2,3,294,1764,230,2023-12-24,2025-06-28,1761.6,0.136240
8,2,4,293,1758,237,2024-03-24,2025-06-29,1761.6,-0.204360
9,2,5,294,1764,264,2024-08-01,2025-06-29,1761.6,0.136240


## 5. Freeze the assignment and inspect deletion locality

The provider assumption, allowed and excluded inputs, hospital pairing, shard balance, slice ranges and assignment hashes are saved before the deletion-membership file is read.

In [23]:
# Fingerprint and freeze the assignment before loading deletion membership.
assignment_hashes = {"shard_assignment_sha256":shard_assignment_hash,
                     "slice_assignment_sha256":slice_assignment_hash}
(RESULT_DIR / "assignment_hashes.json").write_text(json.dumps(assignment_hashes,indent=2)+"\n")
pre_deletion_audit = {
    "status":"frozen_before_deletion_masks", "timestamp_utc":datetime.now(timezone.utc).isoformat(),
    "assumption":"Hospitals are predeclared data-provider boundaries; exact future forget sets are unknown.",
    "slice_assumption":"Chronological data-arrival order within provider-aware shards.",
    "input_columns_used":allowed_assignment_columns,
    "forbidden_columns_not_used":forbidden_assignment_columns,
    "deletion_masks_loaded":deletion_masks_loaded, "assignment_seed":SEED,
    "hospital_pairing":hospital_pairing.to_dict("records"),
    "shard_balance":shard_summary.to_dict("records"),
    "slice_ranges":slice_summary.astype({"minimum_date":"string","maximum_date":"string"}).to_dict("records"),
    "assignment_hashes":assignment_hashes,
}


In [24]:
audit_path=RESULT_DIR/"pre_deletion_assumption_audit.json"
audit_path.write_text(json.dumps(pre_deletion_audit,indent=2,default=str)+"\n")
assert json.loads(audit_path.read_text())["deletion_masks_loaded"] is False

After the saved audit confirms that deletion masks were unavailable during assignment, the five unchanged scenarios from Notebook 02 are loaded. Full deletion membership remains distinct from the training-forget rows that could have influenced model fitting.

In [25]:
# Load exact deletion membership only after provider assignment is frozen.
membership = pd.read_csv(MEMBERSHIP_PATH)
deletion_masks_loaded = True
expected_forget={"recipient_withdrawal":426,"donor_withdrawal":1992,"hospital_removal":4314,
                 "invalid_consent":4148,"retention_expiry":6262}
assert not membership.duplicated(["scenario","assessment_id"]).any()
training_membership=membership.loc[membership.membership_type.eq("training_forget")]
assert training_membership.groupby("scenario").size().to_dict()==expected_forget
scenario_frames={}


In [26]:
# Keep shard and slice operations explicit so SISA provenance remains auditable.
for scenario in SCENARIOS:
    sm=membership.loc[membership.scenario.eq(scenario)]
    parts={}
    for split_name,deleted_name,retained_name in [("train","training_forget","retained_train"),
            ("validation","deleted_validation","retained_validation"),("test","deleted_test","retained_test")]:
        base=assessments.loc[assessments.original_split.eq(split_name)]
        deleted=set(sm.loc[sm.membership_type.eq(deleted_name),"assessment_id"])
        mask=base.assessment_id.isin(deleted)
        parts[deleted_name],parts[retained_name]=base.loc[mask].copy(),base.loc[~mask].copy()
    scenario_frames[scenario]=parts

In [27]:
# Reconstruct the same retained and forget subsets used by earlier methods.
scenario_counts=[]
for scenario in SCENARIOS:
    sm=membership.loc[membership.scenario.eq(scenario)]
    parts=scenario_frames[scenario]
    scenario_counts.append({"scenario":scenario,"complete_deletion_rows":len(sm),
        "training_forget_rows":len(parts["training_forget"]),
        "training_recipients":parts["training_forget"].recipient_id.nunique(),
        "retained_training_rows":len(parts["retained_train"]),
        "retained_validation_rows":len(parts["retained_validation"]),
        "retained_test_rows":len(parts["retained_test"])})
scenario_counts=pd.DataFrame(scenario_counts)
display(scenario_counts)
assert scenario_counts.set_index("scenario").loc["recipient_withdrawal","training_recipients"]==71

,scenario,complete_deletion_rows,training_forget_rows,training_recipients,retained_training_rows,retained_validation_rows,retained_test_rows
0,recipient_withdrawal,600,426,71,41598,8904,8898
1,donor_withdrawal,3000,1992,332,40032,8472,8496
2,hospital_removal,6000,4314,719,37710,8166,8124
3,invalid_consent,6000,4148,2640,37876,8046,8078
4,retention_expiry,9000,6262,2576,35762,7624,7614


The frozen assignment is joined to each training forget set to count affected shards and locate the earliest affected slice. These locality results cannot feed back into shard or slice placement.

In [28]:
# Find the earliest affected slice in each provider-aware shard.
def assignment_impact(current_assignment):
    mapping=current_assignment.set_index("recipient_id")[["component_id","shard_id","slice_id"]]
    rows=[]
    for scenario in SCENARIOS:
        forget=scenario_frames[scenario]["training_forget"].join(mapping,on="recipient_id")
        for shard in range(1,6):
            part=forget.loc[forget.shard_id.eq(shard)]
            slices=sorted(part.slice_id.dropna().astype(int).unique())
            earliest=min(slices) if slices else None
            rows.append({"scenario":scenario,"shard_id":shard,"affected":bool(slices),
                "affected_slices":"|".join(map(str,slices)),"earliest_affected_slice":earliest,
                "forget_rows":len(part),"donor_components_affected":part.component_id.nunique(),
                "hospitals_affected":part.hospital_id.nunique(),
                "replayed_stages":0 if earliest is None else 6-earliest})
    return pd.DataFrame(rows)


In [29]:
scenario_shard_impact=assignment_impact(provider_assignment)

In [30]:
# Measure localisation before interpreting any runtime difference.
locality_rows=[]
for scenario in SCENARIOS:
    impact=scenario_shard_impact.loc[scenario_shard_impact.scenario.eq(scenario)]
    forget=scenario_frames[scenario]["training_forget"]
    earliest=" | ".join(f"S{int(r.shard_id)}:{int(r.earliest_affected_slice)}" for r in impact.itertuples() if r.affected)
    locality_rows.append({"scenario":scenario,"forget_rows":len(forget),
        "recipients":forget.recipient_id.nunique(),"affected_shards":int(impact.affected.sum()),
        "reusable_shards":int((~impact.affected).sum()),"earliest_slices":earliest,
        "replayed_stages":int(impact.replayed_stages.sum()),
        "donor_components_affected":forget.donor_id.nunique(),"hospitals_affected":forget.hospital_id.nunique()})
locality_table=pd.DataFrame(locality_rows)
locality_table.to_csv(RESULT_DIR/"locality_table.csv",index=False)
scenario_shard_impact.to_csv(RESULT_DIR/"scenario_shard_impact.csv",index=False)
display(locality_table)

,scenario,forget_rows,recipients,affected_shards,reusable_shards,earliest_slices,replayed_stages,donor_components_affected,hospitals_affected
0,recipient_withdrawal,426,71,5,0,S1:1 | S2:1 | S3:1 | S4:1 | S5:1,25,71,10
1,donor_withdrawal,1992,332,5,0,S1:1 | S2:1 | S3:1 | S4:1 | S5:1,25,166,10
2,hospital_removal,4314,719,5,0,S1:1 | S2:1 | S3:1 | S4:1 | S5:1,25,708,1
3,invalid_consent,4148,2640,5,0,S1:1 | S2:1 | S3:1 | S4:1 | S5:1,25,2438,10
4,retention_expiry,6262,2576,5,0,S1:1 | S2:1 | S3:1 | S4:1 | S5:1,25,2383,10


All five tested requests affected all five provider-aware shards and began at slice 1, so the assignment did not reduce replay work in these scenarios. The result is retained rather than optimising the partition after seeing the requests.

## 6. Train the provider-aware SISA ensemble

Five independent shard models use the established architecture, optimiser, preprocessor and validation method. Each checkpoint contains all provider-shard data available through its slice, and the five final probabilities are averaged with equal weight.

In [31]:
# Keep the baseline architecture unchanged across all constituent models.
class BaselineMLP(nn.Module):
    def __init__(self,input_dim=24,hidden_dims=(64,32),dropout=.10):
        super().__init__()
        self.network=nn.Sequential(nn.Linear(input_dim,hidden_dims[0]),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(hidden_dims[0],hidden_dims[1]),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden_dims[1],1))
    def forward(self,x): return self.network(x).squeeze(1)

probe=BaselineMLP()
assert sum(p.numel() for p in probe.parameters())==3713
validation_frame=assessments.loc[assessments.original_split.eq("validation")]


In [32]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.

def predict(model,matrix,batch=2048):
    model.eval()
    arrays=[]
    with torch.no_grad():
        for start in range(0,len(matrix),batch):
            arrays.append(torch.sigmoid(model(torch.from_numpy(matrix[start:start+batch]))).numpy())
    return np.concatenate(arrays)

In [33]:
# Use validation loss without updating model parameters.
def weighted_loss(model,matrix,labels,class_weight):
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([class_weight],dtype=torch.float32))
    model.eval()
    total=0.0
    with torch.no_grad():
        for start in range(0,len(matrix),2048):
            x=torch.from_numpy(matrix[start:start+2048])
            y=torch.from_numpy(labels[start:start+2048])
            total+=float(criterion(model(x),y))*len(x)
    return total/len(matrix)

def row_fingerprint(frame):
    return hashlib.sha256("\n".join(sorted(frame.assessment_id.astype(str))).encode()).hexdigest()


In [34]:
# Keep shard and slice operations explicit so SISA provenance remains auditable.

def positive_class_weight(frame):
    positive_count=int(frame[target].eq(1).sum())
    negative_count=int(frame[target].eq(0).sum())
    assert positive_count>0 and negative_count>0
    return float(negative_count/positive_count)

def slice_training_seed(shard_id,slice_id):
    return derived_seed("provider_slice_training",shard_id,slice_id)

In [35]:
# Train each stage with deterministic ordering and retained validation.
def train_slice(model,optimizer,frame,validation,class_weight,seed):
    reset_seed(seed)
    x=preprocessor.transform(frame[features]).astype("float32")
    y=frame[target].to_numpy(dtype="float32")
    vx=preprocessor.transform(validation[features]).astype("float32")
    vy=validation[target].to_numpy(dtype="float32")
    loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y)),batch_size=BATCH_SIZE,
        shuffle=True,generator=torch.Generator().manual_seed(seed))
    criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([class_weight],dtype=torch.float32))
    best_model,best_optimizer,best_loss,best_epoch=None,None,np.inf,0
    patience=0
    history=[]
    for epoch in range(1,MAX_EPOCHS_PER_SLICE+1):
        model.train()
        total=0.0
        for inputs,labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(inputs),labels)
            loss.backward()
            optimizer.step()
            total+=float(loss.detach())*len(inputs)
        val_loss=weighted_loss(model,vx,vy,class_weight)
        improved=val_loss<best_loss-1e-8
        history.append((epoch,total/len(x),val_loss,improved))
        if improved:
            best_loss,best_epoch=val_loss,epoch
            best_model=copy.deepcopy(model.state_dict())
            best_optimizer=copy.deepcopy(optimizer.state_dict())
            patience=0
        else: patience+=1
        if patience>=EARLY_STOPPING_PATIENCE: break
    model.load_state_dict(best_model)
    optimizer.load_state_dict(best_optimizer)
    return history,best_epoch,best_loss

In [36]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
def aggregate_models(models,matrix):
    arrays=[predict(model,matrix).astype(np.float64) for model in models]
    assert len(arrays)==5 and len({len(a) for a in arrays})==1
    return np.mean(np.stack(arrays),axis=0)


In [37]:

def train_original_shard(shard_id):
    shard=train_frame.loc[train_frame.shard_id.eq(shard_id)]
    init_seed=derived_seed("provider_original",shard_id)
    reset_seed(init_seed)
    model=BaselineMLP()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    histories,summaries=[],[]
    for slice_id in range(1,6):
        # Train each checkpoint on all provider-shard data available so far.
        frame=shard.loc[shard.slice_id.le(slice_id)]
        class_weight=positive_class_weight(frame)
        seed=slice_training_seed(shard_id,slice_id)
        started=time.perf_counter()
        assert frame.slice_id.max()<=slice_id
        history,best_epoch,best_loss=train_slice(model,optimizer,frame,validation_frame,class_weight,seed)
        elapsed=time.perf_counter()-started
        for epoch,train_loss,val_loss,improved in history:
            histories.append({"shard_id":shard_id,"slice_id":slice_id,"epoch":epoch,"seed":seed,
                "training_weighted_bce":train_loss,"validation_weighted_bce":val_loss,"best":improved})
        checkpoint={"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),
            "shard_id":shard_id,"slice_id":slice_id,"initialization_seed":init_seed,
            "slice_training_seed":seed,"class_weight":class_weight,"features":features,
            "cumulative_training_rows":len(frame),"training_row_fingerprint":row_fingerprint(frame)}
        path=ORIGINAL_DIR/f"shard_{shard_id}_slice_{slice_id}.pt"
        torch.save(checkpoint,path)
        summaries.append({"shard_id":shard_id,"slice_id":slice_id,"rows":len(frame),
            "new_slice_rows":int(shard.slice_id.eq(slice_id).sum()),"class_weight":class_weight,
            "training_row_fingerprint":row_fingerprint(frame),"epochs":len(history),
            "best_epoch":best_epoch,"best_validation_weighted_bce":best_loss,"training_seconds":elapsed,
            "checkpoint_path":path.relative_to(ROOT).as_posix()})
    final_path=ORIGINAL_DIR/f"shard_{shard_id}_final.pt"
    torch.save(checkpoint,final_path)
    return model.eval(),histories,summaries,final_path

In [38]:
# Save all cumulative checkpoints needed for later rollback.
original_models=[]
original_history_rows=[]
original_summary_rows=[]
final_paths=[]
preparation_started=time.perf_counter()
for shard_id in range(1,6):
    model,history,summary,final_path=train_original_shard(shard_id)
    original_models.append(model)
    original_history_rows.extend(history)
    original_summary_rows.extend(summary)
    final_paths.append(final_path)
preparation_seconds=time.perf_counter()-preparation_started
original_training_history=pd.DataFrame(original_history_rows)
original_training_summary=pd.DataFrame(original_summary_rows)
original_stage_audit_rows=[]


In [39]:
# Preserve cumulative slice checkpoints so later deletion replay is auditable.
for row in original_training_summary.itertuples():
    expected=train_frame.loc[train_frame.shard_id.eq(row.shard_id)&train_frame.slice_id.le(row.slice_id)]
    checkpoint=torch.load(ORIGINAL_DIR/f"shard_{row.shard_id}_slice_{row.slice_id}.pt",weights_only=False)
    expected_weight=positive_class_weight(expected)
    expected_fingerprint=row_fingerprint(expected)
    passed=(row.rows==len(expected) and row.training_row_fingerprint==expected_fingerprint
        and checkpoint["cumulative_training_rows"]==len(expected)
        and checkpoint["training_row_fingerprint"]==expected_fingerprint
        and np.isclose(row.class_weight,expected_weight) and np.isclose(checkpoint["class_weight"],expected_weight)
        and expected.slice_id.max()<=row.slice_id)
    original_stage_audit_rows.append({"shard_id":row.shard_id,"slice_id":row.slice_id,
        "cumulative_training_rows":len(expected),"maximum_training_slice":int(expected.slice_id.max()),
        "future_slice_rows":int(expected.slice_id.gt(row.slice_id).sum()),
        "class_weight":row.class_weight,"expected_class_weight":expected_weight,"pass":bool(passed)})
original_cumulative_stage_audit=pd.DataFrame(original_stage_audit_rows)


In [40]:
# Keep shard and slice operations explicit so SISA provenance remains auditable.
original_cumulative_stage_pass=(len(original_cumulative_stage_audit)==N_SHARDS*N_SLICES
                                and bool(original_cumulative_stage_audit["pass"].all()))
assert original_cumulative_stage_pass
original_manifest={"ensemble":"provider_original","aggregation":"unweighted_mean","models":[
    {"shard_id":i,"source":"provider_original","model_path":p.relative_to(ROOT).as_posix()}
    for i,p in enumerate(final_paths,start=1)]}
(ORIGINAL_DIR/"ensemble_manifest.json").write_text(json.dumps(original_manifest,indent=2)+"\n")
provider_configuration={"experiment":"provider-aware SISA","n_shards":5,"n_slices":5,"seed":42,
    "threshold":THRESHOLD,"architecture":[24,64,32,1],"batch_size":BATCH_SIZE,
    "learning_rate":LEARNING_RATE,"weight_decay":WEIGHT_DECAY,"maximum_epochs_per_slice":MAX_EPOCHS_PER_SLICE,
    "early_stopping_patience":EARLY_STOPPING_PATIENCE,"assignment_inputs":allowed_assignment_columns,
    "slice_training_data":"cumulative shard rows with slice_id <= current slice",
    "class_weight_scope":"same cumulative rows used at each stage",
    "deletion_membership_used_for_assignment":False}
(ORIGINAL_DIR/"configuration.json").write_text(json.dumps(provider_configuration,indent=2)+"\n")


633

In [41]:
assert len(list(ORIGINAL_DIR.glob("shard_*_slice_*.pt")))==25

In [42]:
# Establish original ensemble predictions before deletion replay.
original_prediction_rows=[]
for partition,frame in [("validation",validation_frame),("test",assessments.loc[assessments.original_split.eq("test")])]:
    matrix=preprocessor.transform(frame[features]).astype("float32")
    probability=aggregate_models(original_models,matrix)
    original_prediction_rows.append(pd.DataFrame({"partition":partition,"evaluation_row":np.arange(len(frame)),
        "assessment_id":frame.assessment_id.to_numpy(),"true_target":frame[target].to_numpy(),
        "provider_original_probability":probability,"provider_original_prediction":(probability>=THRESHOLD).astype(int)}))
original_ensemble_predictions=pd.concat(original_prediction_rows,ignore_index=True)
display(Markdown(f"Initial provider-aware SISA trained 25 slice checkpoints in **{preparation_seconds:.3f}s**."))

Initial provider-aware SISA trained 25 slice checkpoints in **3.690s**.

## 7. Apply deletion requests and replay affected shards

For each affected shard, replay restores the last unaffected checkpoint or uses the same deterministic initialisation when slice 1 is affected. Forgotten training rows are removed before cumulative replay, future-slice rows remain excluded from earlier stages, and class weights are recalculated from the retained cumulative rows.

In [43]:
# Replay only the affected shard suffix while preserving the frozen assignment.
def restore_for_replay(shard_id,earliest):
    init_seed=derived_seed("provider_original",shard_id)
    reset_seed(init_seed)
    model=BaselineMLP()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    if earliest>1:
        checkpoint=torch.load(ORIGINAL_DIR/f"shard_{shard_id}_slice_{earliest-1}.pt",weights_only=False)
        assert checkpoint["slice_id"]==earliest-1
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return model,optimizer,init_seed


In [44]:

def replay_shard(scenario,shard_id,impact,forget_ids):
    earliest=int(impact.earliest_affected_slice)
    model,optimizer,init_seed=restore_for_replay(shard_id,earliest)
    # Exclude forgotten training rows before cumulative replay.
    retained=train_frame.loc[train_frame.shard_id.eq(shard_id)&~train_frame.assessment_id.isin(forget_ids)]
    validation=scenario_frames[scenario]["retained_validation"]
    started=time.perf_counter()
    rows=[]
    class_weights={}
    cumulative_rows={}
    for slice_id in range(earliest,6):
        # Replay on all retained provider-shard data available so far.
        frame=retained.loc[retained.slice_id.le(slice_id)]
        if frame.empty: continue
        class_weight=positive_class_weight(frame)
        class_weights[slice_id]=class_weight
        cumulative_rows[slice_id]=len(frame)
        seed=slice_training_seed(shard_id,slice_id)
        history,best_epoch,best_loss=train_slice(model,optimizer,frame,validation,class_weight,seed)
        expected=train_frame.loc[train_frame.shard_id.eq(shard_id)&train_frame.slice_id.le(slice_id)
            &~train_frame.assessment_id.isin(forget_ids)]
        expected_weight=positive_class_weight(expected)
        passed=(row_fingerprint(frame)==row_fingerprint(expected) and frame.slice_id.max()<=slice_id
            and not frame.assessment_id.isin(forget_ids).any() and np.isclose(class_weight,expected_weight))
        replay_stage_audit_rows.append({"scenario":scenario,"shard_id":shard_id,"slice_id":slice_id,
            "cumulative_retained_rows":len(frame),"maximum_training_slice":int(frame.slice_id.max()),
            "future_slice_rows":int(frame.slice_id.gt(slice_id).sum()),
            "forgotten_rows_present":int(frame.assessment_id.isin(forget_ids).sum()),
            "class_weight":class_weight,"expected_class_weight":expected_weight,"pass":bool(passed)})
        for epoch,train_loss,val_loss,improved in history:
            rows.append({"scenario":scenario,"shard_id":shard_id,"slice_id":slice_id,"epoch":epoch,
                "seed":seed,"training_weighted_bce":train_loss,"validation_weighted_bce":val_loss,"best":improved})
    summary={"scenario":scenario,"shard_id":shard_id,"earliest_affected_slice":earliest,
        "replayed_stages":6-earliest,"forget_rows":int(impact.forget_rows),"retained_rows":len(retained),
        "training_seconds":time.perf_counter()-started,"initialization_seed":init_seed,
        "slice_class_weights":json.dumps(class_weights,sort_keys=True),
        "cumulative_rows_by_slice":json.dumps(cumulative_rows,sort_keys=True)}
    return model.eval(),optimizer,rows,summary

In [45]:
# Reuse unaffected models and replay only affected shard suffixes.
scenario_models={}
scenario_manifests={}
replay_history_rows=[]
replay_summary_rows=[]
replay_stage_audit_rows=[]


In [46]:
# Replay only the affected shard suffix while preserving the frozen assignment.
for scenario in SCENARIOS:
    scenario_dir=UPDATED_DIR/scenario
    scenario_dir.mkdir(parents=True,exist_ok=True)
    forget_ids=set(scenario_frames[scenario]["training_forget"].assessment_id)
    models=[]
    entries=[]
    for shard_id in range(1,6):
        impact=scenario_shard_impact.loc[(scenario_shard_impact.scenario.eq(scenario))&
                                         (scenario_shard_impact.shard_id.eq(shard_id))].iloc[0]
        if not impact.affected:
            path=ORIGINAL_DIR/f"shard_{shard_id}_final.pt"
            checkpoint=torch.load(path,weights_only=False)
            model=BaselineMLP()
            model.load_state_dict(checkpoint["model_state_dict"])
            entries.append({"shard_id":shard_id,"source":"reused_original","model_path":path.relative_to(ROOT).as_posix()})
        else:
            model,optimizer,history,summary=replay_shard(scenario,shard_id,impact,forget_ids)
            path=scenario_dir/f"shard_{shard_id}_model.pt"
            torch.save({"model_state_dict":model.state_dict(),"optimizer_state_dict":optimizer.state_dict(),
                "scenario":scenario,"shard_id":shard_id,"forgotten_ids_sha256":hashlib.sha256("\n".join(sorted(forget_ids)).encode()).hexdigest()},path)
            replay_history_rows.extend(history)
            replay_summary_rows.append(summary)
            entries.append({"shard_id":shard_id,"source":"updated_scenario","model_path":path.relative_to(ROOT).as_posix()})
        models.append(model.eval())
    manifest={"scenario":scenario,"aggregation":"unweighted_mean","models":entries}
    (scenario_dir/"ensemble_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
    scenario_models[scenario],scenario_manifests[scenario]=models,manifest


In [47]:
# Replay only the affected shard suffix while preserving the frozen assignment.
replay_history=pd.DataFrame(replay_history_rows)
replay_summary=pd.DataFrame(replay_summary_rows)
replay_cumulative_stage_audit=pd.DataFrame(replay_stage_audit_rows)
expected_replay_stages=int(scenario_shard_impact.replayed_stages.sum())
replay_cumulative_stage_pass=(len(replay_cumulative_stage_audit)==expected_replay_stages
                              and bool(replay_cumulative_stage_audit["pass"].all()))
assert replay_cumulative_stage_pass

In [48]:
# Aggregate updated ensembles on aligned retained and forget rows.
scenario_probabilities={}
for scenario in SCENARIOS:
    scenario_probabilities[scenario]={}
    for partition in ["retained_test","training_forget"]:
        frame=scenario_frames[scenario][partition]
        matrix=preprocessor.transform(frame[features]).astype("float32")
        scenario_probabilities[scenario][partition]=aggregate_models(scenario_models[scenario],matrix)

deletion_seconds=replay_summary.groupby("scenario").training_seconds.sum().reindex(SCENARIOS)
for scenario in SCENARIOS:
    audit=replay_cumulative_stage_audit.loc[replay_cumulative_stage_audit.scenario.eq(scenario)]
    assert audit.forgotten_rows_present.sum()==0 and audit.future_slice_rows.sum()==0 and audit["pass"].all()
display(pd.DataFrame({"scenario":SCENARIOS,"provider_update_seconds":deletion_seconds.values}))

,scenario,provider_update_seconds
0,recipient_withdrawal,3.593850
1,donor_withdrawal,3.451578
2,hospital_removal,3.445945
3,invalid_consent,3.341944
4,retention_expiry,3.236781


## 8. Evaluate retained utility and forgetting

Each updated ensemble is evaluated on the exact scenario-specific retained test rows. Threshold-dependent metrics retain the frozen threshold of 0.62, while BCE evaluates probability quality independently of that threshold.

In [49]:
# Keep threshold 0.62 fixed for retained-utility comparisons.
def binary_metrics(y,p,threshold=THRESHOLD):
    pred=(p>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"n":len(y),"positive_count":int(y.sum()),"prevalence":float(y.mean()),"threshold":threshold,
        "pr_auc":average_precision_score(y,p),"auroc":roc_auc_score(y,p),
        "balanced_accuracy":balanced_accuracy_score(y,pred),"f1":f1_score(y,pred,zero_division=0),
        "binary_cross_entropy":log_loss(y,p,labels=[0,1]),"precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),"specificity":tn/(tn+fp),"accuracy":accuracy_score(y,pred),
        "positive_prediction_rate":float(pred.mean()),"true_negative":int(tn),"false_positive":int(fp),
        "false_negative":int(fn),"true_positive":int(tp)}

def harmonic(values,epsilon=1e-12):
    values=np.asarray(values,dtype=float)
    return float(len(values)/np.sum(1/np.maximum(values,epsilon)))

In [50]:
# Evaluate utility only on scenario-specific retained test records.
prediction_frames=[]
utility_rows=[]


In [51]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
for scenario in SCENARIOS:
    frame=scenario_frames[scenario]["retained_test"].reset_index(drop=True)
    y=frame[target].to_numpy()
    matrix=preprocessor.transform(frame[features]).astype("float32")
    original=aggregate_models(original_models,matrix)
    updated=scenario_probabilities[scenario]["retained_test"]
    prediction_frames.append(pd.DataFrame({"scenario":scenario,"evaluation_row":np.arange(len(frame)),
        "assessment_id":frame.assessment_id,"true_target":y,"provider_original_probability":original,
        "provider_aware_probability":updated,"provider_original_prediction":(original>=THRESHOLD).astype(int),
        "provider_aware_prediction":(updated>=THRESHOLD).astype(int)}))
    for model,probability in [("provider_original_sisa",original),("provider_aware_sisa",updated)]:
        row={"scenario":scenario,"model":model,**binary_metrics(y,probability)}
        row["bce_quality"]=np.exp(-row["binary_cross_entropy"])
        row["composite_model_utility"]=harmonic([row[k] for k in ["pr_auc","auroc","balanced_accuracy","f1","bce_quality"]])
        utility_rows.append(row)


In [52]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
retained_test_predictions=pd.concat(prediction_frames,ignore_index=True)
utility_metrics=pd.DataFrame(utility_rows)
display(utility_metrics.loc[utility_metrics.model.eq("provider_aware_sisa"),
    ["scenario","pr_auc","auroc","balanced_accuracy","f1","binary_cross_entropy","composite_model_utility"]])

,scenario,pr_auc,auroc,balanced_accuracy,f1,binary_cross_entropy,composite_model_utility
1,recipient_withdrawal,0.169787,0.722393,0.618549,0.229354,0.582722,0.332409
3,donor_withdrawal,0.172668,0.723813,0.619519,0.233180,0.577538,0.336533
5,hospital_removal,0.172322,0.729662,0.624876,0.238749,0.584291,0.338842
7,invalid_consent,0.160315,0.722867,0.617024,0.219822,0.578358,0.321047
9,retention_expiry,0.156761,0.728733,0.621072,0.228234,0.577199,0.322081


Retained utility is reported separately from structural replay work. Forgetting is then evaluated on the established training-forget rows using the unchanged Truth Ratio definition and scenario-specific full retraining as the behavioural reference.

Lower KS means the observed Truth Ratio distributions are closer. Its p-value is sample-size-sensitive and does not by itself prove successful or failed unlearning; SISA is also an ensemble whereas full retraining is a monolithic model.

In [53]:
# Use full retraining as the behavioural forgetting reference.
full_forget=pd.read_csv(FULL_RESULT_DIR/"forget_set_reference_predictions.csv")
forget_prediction_frames=[]
truth_rows=[]
quality_rows=[]


In [54]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
for scenario in SCENARIOS:
    frame=scenario_frames[scenario]["training_forget"].reset_index(drop=True)
    y=frame[target].to_numpy()
    provider=scenario_probabilities[scenario]["training_forget"]
    reference=full_forget.loc[full_forget.scenario.eq(scenario)].reset_index(drop=True)
    assert list(frame.assessment_id)==list(reference.assessment_id)
    full=reference.full_retraining_probability.to_numpy(dtype=float)
    output=pd.DataFrame({"scenario":scenario,"evaluation_row":np.arange(len(frame)),
        "assessment_id":frame.assessment_id,"true_target":y,
        "provider_aware_probability":provider,"full_retraining_probability":full})
    forget_prediction_frames.append(output)
    scenario_truth=[]
    for model,p in [("provider_aware_sisa",provider),("full_retraining",full)]:
        p_true=np.where(y==1,p,1-p)
        p_incorrect=np.where(y==1,1-p,p)
        ratio=(p_incorrect+EPSILON)/(p_true+EPSILON)
        part=pd.DataFrame({"scenario":scenario,"assessment_id":frame.assessment_id,"model":model,
            "p_true":p_true,"p_incorrect":p_incorrect,"epsilon":EPSILON,"truth_ratio":ratio})
        truth_rows.append(part)
        scenario_truth.append((model,ratio))
    ks=ks_2samp(scenario_truth[0][1],scenario_truth[1][1])
    quality_rows.append({"scenario":scenario,"forget_rows":len(frame),"ks_statistic":ks.statistic,
        "ks_p_value":ks.pvalue,"provider_truth_ratio_mean":scenario_truth[0][1].mean(),
        "full_retraining_truth_ratio_mean":scenario_truth[1][1].mean()})


In [55]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
forget_set_predictions=pd.concat(forget_prediction_frames,ignore_index=True)
truth_ratio_values=pd.concat(truth_rows,ignore_index=True)
forget_quality=pd.DataFrame(quality_rows)
display(forget_quality)

,scenario,forget_rows,ks_statistic,ks_p_value,provider_truth_ratio_mean,full_retraining_truth_ratio_mean
0,recipient_withdrawal,426,0.143192,3.155033e-04,0.907717,0.951461
1,donor_withdrawal,1992,0.105924,3.791481e-10,0.990993,1.065314
2,hospital_removal,4314,0.130274,2.617446e-32,0.961153,1.011090
3,invalid_consent,4148,0.025796,1.265391e-01,1.243922,1.261086
4,retention_expiry,6262,0.070584,5.515370e-14,1.508805,1.442658


In [56]:
# Summarise Truth Ratio distributions without reducing them to one threshold.
truth_ratio_summary=truth_ratio_values.groupby(["scenario","model"]).truth_ratio.agg(
    ["count","mean","std","min","median","max"]).reset_index()
assert not forget_quality.ks_p_value.isna().any()

## 9. Compare localisation and runtime with standard SISA

Provider-aware runtime is compared with standard SISA, Policy-Aware SISA and full retraining only after affected shards and replay stages are counted. Wall-clock results are descriptive: they depend on shard sizes, cumulative training, early stopping, hardware and normal execution variation.

In [57]:
# Compare runtime only after counting affected shards and replay stages.
standard_eff=pd.read_csv(STANDARD_RESULT_DIR/"efficiency_comparison.csv").set_index("scenario")
policy_eff=pd.read_csv(POLICY_RESULT_DIR/"efficiency_comparison.csv").set_index("scenario")
full_summary=json.loads((FULL_RESULT_DIR/"run_summary.json").read_text())
efficiency_rows=[]
for scenario in SCENARIOS:
    provider=float(deletion_seconds.loc[scenario])
    standard=float(standard_eff.loc[scenario,"selective_retraining_seconds"])
    policy=float(policy_eff.loc[scenario,"policy_aware_sequential_seconds"])
    full=float(full_summary["training_seconds"][scenario])
    loc=locality_table.set_index("scenario").loc[scenario]
    efficiency_rows.append({"scenario":scenario,"provider_aware_seconds":provider,
        "standard_sisa_seconds":standard,"policy_aware_seconds":policy,"full_retraining_seconds":full,
        "speed_up_vs_full_retraining":full/provider,"speed_up_vs_standard_sisa":standard/provider,
        "time_reduction_pct_vs_full":100*(full-provider)/full,
        "affected_shards":int(loc.affected_shards),"replayed_stages":int(loc.replayed_stages)})


In [58]:
efficiency_comparison=pd.DataFrame(efficiency_rows)
display(efficiency_comparison)

,scenario,provider_aware_seconds,standard_sisa_seconds,policy_aware_seconds,full_retraining_seconds,speed_up_vs_full_retraining,speed_up_vs_standard_sisa,time_reduction_pct_vs_full,affected_shards,replayed_stages
0,recipient_withdrawal,3.593850,3.638856,0.063902,3.923029,1.091595,1.012523,8.390925,5,25
1,donor_withdrawal,3.451578,3.631015,0.063300,3.450068,0.999563,1.051987,-0.043760,5,25
2,hospital_removal,3.445945,3.454197,0.283568,3.847863,1.116635,1.002395,10.445233,5,25
3,invalid_consent,3.341944,3.449524,1.041479,3.528170,1.055724,1.032191,5.278247,5,25
4,retention_expiry,3.236781,3.498488,0.888951,3.111061,0.961159,1.080854,-4.041080,5,25


The three SISA configurations are aligned by scenario. Standard SISA is request-agnostic, Provider-Aware SISA uses pre-request provider and relational information, and Policy-Aware SISA uses exact predefined membership as an optimistic sensitivity analysis.

Structural localisation, retained utility, forgetting similarity and runtime remain separate comparison dimensions.

In [59]:
# Compare partition strategies without changing their saved results.
standard_utility=pd.read_csv(STANDARD_RESULT_DIR/"composite_model_utility.csv")
policy_utility=pd.read_csv(POLICY_RESULT_DIR/"composite_model_utility.csv")
standard_quality=pd.read_csv(STANDARD_RESULT_DIR/"forget_quality.csv").set_index("scenario")
policy_quality=pd.read_csv(POLICY_RESULT_DIR/"forget_quality.csv").set_index("scenario")
provider_utility=utility_metrics.loc[utility_metrics.model.eq("provider_aware_sisa")].set_index("scenario")
provider_quality=forget_quality.set_index("scenario")
comparison_rows=[]


In [60]:
# Replay only the affected shard suffix while preserving the frozen assignment.
for scenario in SCENARIOS:
    configurations=[
        ("standard_hash_based",standard_utility.loc[(standard_utility.scenario.eq(scenario))&standard_utility.model.eq("updated_sisa"),"composite_model_utility"].iloc[0],standard_quality.loc[scenario,"ks_statistic"],standard_quality.loc[scenario,"tofu_style_forget_quality"],standard_eff.loc[scenario,"selective_retraining_seconds"],5,25),
        ("provider_aware",provider_utility.loc[scenario,"composite_model_utility"],provider_quality.loc[scenario,"ks_statistic"],provider_quality.loc[scenario,"ks_p_value"],efficiency_comparison.set_index("scenario").loc[scenario,"provider_aware_seconds"],locality_table.set_index("scenario").loc[scenario,"affected_shards"],locality_table.set_index("scenario").loc[scenario,"replayed_stages"]),
        ("exact_membership_policy_aware",policy_utility.loc[(policy_utility.scenario.eq(scenario))&policy_utility.model.eq("policy_aware_sisa"),"composite_model_utility"].iloc[0],policy_quality.loc[scenario,"ks_statistic"],policy_quality.loc[scenario,"tofu_style_forget_quality"],policy_eff.loc[scenario,"policy_aware_sequential_seconds"],policy_eff.loc[scenario,"affected_shards"],policy_eff.loc[scenario,"replayed_slice_stages"])]
    for configuration,utility,ks,pvalue,runtime,affected,replayed in configurations:
        comparison_rows.append({"scenario":scenario,"configuration":configuration,"utility":utility,
            "ks_statistic":ks,"ks_p_value":pvalue,"runtime_seconds":runtime,
            "affected_shards":int(affected),"replayed_stages":int(replayed)})
sisa_configuration_comparison=pd.DataFrame(comparison_rows)
sisa_mean_summary=sisa_configuration_comparison.groupby("configuration",as_index=False).agg(
    mean_utility=("utility","mean"),mean_ks_statistic=("ks_statistic","mean"),mean_runtime_seconds=("runtime_seconds","mean"))
display(sisa_configuration_comparison)
display(sisa_mean_summary)

,scenario,configuration,utility,ks_statistic,ks_p_value,runtime_seconds,affected_shards,replayed_stages
0,recipient_withdrawal,standard_hash_based,0.331071,0.150235,1.301308e-04,3.638856,5,25
1,recipient_withdrawal,provider_aware,0.332409,0.143192,3.155033e-04,3.593850,5,25
2,recipient_withdrawal,exact_membership_policy_aware,0.326154,0.206573,2.283404e-08,0.063902,1,1
3,donor_withdrawal,standard_hash_based,0.327993,0.107430,1.994542e-10,3.631015,5,25
4,donor_withdrawal,provider_aware,0.336533,0.105924,3.791481e-10,3.451578,5,25
5,donor_withdrawal,exact_membership_policy_aware,0.327577,0.177711,6.934800e-28,0.063300,1,2
6,hospital_removal,standard_hash_based,0.332814,0.131896,4.133776e-33,3.454197,5,25
7,hospital_removal,provider_aware,0.338842,0.130274,2.617446e-32,3.445945,5,25
8,hospital_removal,exact_membership_policy_aware,0.331412,0.203292,2.182257e-78,0.283568,1,5
9,invalid_consent,standard_hash_based,0.317833,0.039778,2.819496e-03,3.449524,5,25


,configuration,mean_utility,mean_ks_statistic,mean_runtime_seconds
0,exact_membership_policy_aware,0.324798,0.165967,0.468240
1,provider_aware,0.330183,0.095154,3.414020
2,standard_hash_based,0.327035,0.095417,3.534416


The executed tables are interpreted below without changing the frozen assignment. A lack of improved localisation remains a valid exploratory result.

In [61]:
# Treat the lack of localisation as a valid exploratory result.
provider_mean=sisa_mean_summary.set_index("configuration").loc["provider_aware"]
standard_mean=sisa_mean_summary.set_index("configuration").loc["standard_hash_based"]
benefited=locality_table.loc[locality_table.affected_shards.lt(5),"scenario"].tolist()
not_benefited=locality_table.loc[locality_table.affected_shards.eq(5),"scenario"].tolist()
faster_standard=efficiency_comparison.loc[efficiency_comparison.speed_up_vs_standard_sisa.gt(1),"scenario"].tolist()
faster_full=efficiency_comparison.loc[efficiency_comparison.speed_up_vs_full_retraining.gt(1),"scenario"].tolist()
interpretation=(f"Provider-aware sharding reduced affected-shard count for {len(benefited)} scenario(s): "
    f"{', '.join(benefited) if benefited else 'none'}. It did not reduce shard spread for "
    f"{', '.join(not_benefited) if not_benefited else 'none'}. Chronological slices produced "
    f"{int(locality_table.replayed_stages.sum())} replay stages in total. Provider-aware SISA was faster than "
    f"standard SISA in {len(faster_standard)} scenarios and faster than full retraining in {len(faster_full)}. "
    f"Mean utility was {provider_mean.mean_utility:.6f} versus {standard_mean.mean_utility:.6f}; mean KS was "
    f"{provider_mean.mean_ks_statistic:.6f} versus {standard_mean.mean_ks_statistic:.6f}. "
    f"Donor-linkage constraints caused {cross_hospital_spillover_rows} hospital-spillover rows.")
display(Markdown(interpretation))

Provider-aware sharding reduced affected-shard count for 0 scenario(s): none. It did not reduce shard spread for recipient_withdrawal, donor_withdrawal, hospital_removal, invalid_consent, retention_expiry. Chronological slices produced 125 replay stages in total. Provider-aware SISA was faster than standard SISA in 5 scenarios and faster than full retraining in 3. Mean utility was 0.330183 versus 0.327035; mean KS was 0.095154 versus 0.095417. Donor-linkage constraints caused 6630 hospital-spillover rows.

## 10. Findings, limitations and verification

Provider-Aware SISA used plausible hospital provenance and donor-linked structure available before deletion requests; it did not use exact future policy membership. Preserving donor-linked histories caused hospital spillover, and all five tested requests still affected all five shards from slice 1. Plausible operational structure therefore did not automatically provide selective-retraining savings in this experiment.

Retained utility and Truth Ratio similarity are separate from structural localisation, and measured runtime is only descriptive. Standard hash-based SISA remains the main SISA reference, while Provider-Aware and Policy-Aware SISA remain distinct exploratory configurations.

The final cells retain the full validity checks: assignment timing and integrity, recipient and donor atomicity, chronological slices, cumulative original training and replay, future-slice and forgotten-row exclusion, stage-specific class weights, checkpoint and ensemble reproduction, retained-test alignment, Truth Ratio, KS, required artifacts and protected-input hashes.

In [62]:
# Preserve the tables needed to reproduce assignment, utility and forgetting.
outputs={"component_summary.csv":component_summary,"shard_summary.csv":shard_summary,
    "shard_validation.csv":shard_validation,"hospital_fragmentation.csv":fragmentation,
    "slice_summary.csv":slice_summary,"slice_validation.csv":slice_validation,
    "scenario_counts.csv":scenario_counts,"locality_table.csv":locality_table,
    "scenario_shard_impact.csv":scenario_shard_impact,"original_training_history.csv":original_training_history,
    "original_training_summary.csv":original_training_summary,"original_cumulative_stage_audit.csv":original_cumulative_stage_audit,
    "original_ensemble_predictions.csv":original_ensemble_predictions,
    "selective_retraining_history.csv":replay_history,"selective_retraining_summary.csv":replay_summary,
    "replay_cumulative_stage_audit.csv":replay_cumulative_stage_audit,
    "retained_test_predictions.csv":retained_test_predictions,"utility_metrics.csv":utility_metrics,
    "composite_model_utility.csv":utility_metrics[["scenario","model","pr_auc","auroc","balanced_accuracy","f1","bce_quality","composite_model_utility"]],
    "forget_set_predictions.csv":forget_set_predictions,"truth_ratio_values.csv":truth_ratio_values,
    "truth_ratio_summary.csv":truth_ratio_summary,"forget_quality.csv":forget_quality,
    "efficiency_comparison.csv":efficiency_comparison,"sisa_configuration_comparison.csv":sisa_configuration_comparison,
    "sisa_mean_summary.csv":sisa_mean_summary}


In [63]:
for name,frame in outputs.items(): frame.to_csv(RESULT_DIR/name,index=False,float_format="%.17g")
(RESULT_DIR/"configuration.json").write_text(json.dumps(provider_configuration,indent=2)+"\n")

633

In [64]:
# Reload saved constituent models before checking ensemble reproduction.
def load_manifest_models(manifest):
    models=[]
    for item in manifest["models"]:
        checkpoint=torch.load(ROOT/item["model_path"],weights_only=False)
        model=BaselineMLP()
        model.load_state_dict(checkpoint["model_state_dict"])
        models.append(model.eval())
    return models

reloaded_models={scenario:load_manifest_models(manifest) for scenario,manifest in scenario_manifests.items()}
reload_rows=[]


In [65]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
for scenario in SCENARIOS:
    for partition,column in [("retained_test","provider_aware_probability"),("training_forget","provider_aware_probability")]:
        frame=scenario_frames[scenario][partition]
        matrix=preprocessor.transform(frame[features]).astype("float32")
        reproduced=aggregate_models(reloaded_models[scenario],matrix)
        saved=retained_test_predictions if partition=="retained_test" else forget_set_predictions
        expected=saved.loc[saved.scenario.eq(scenario),column].to_numpy()
        reload_rows.append({"scenario":scenario,"partition":partition,
            "maximum_absolute_difference":float(np.max(np.abs(reproduced-expected))),
            "pass":bool(np.allclose(reproduced,expected,atol=1e-8,rtol=1e-8))})
ensemble_probability_reproduction=pd.DataFrame(reload_rows)
ensemble_probability_reproduction.to_csv(RESULT_DIR/"ensemble_probability_reproduction.csv",index=False)
assert ensemble_probability_reproduction["pass"].all()

In [66]:
# Recalculate utility from persisted probabilities.
saved_predictions=pd.read_csv(RESULT_DIR/"retained_test_predictions.csv")
saved_metrics=pd.read_csv(RESULT_DIR/"utility_metrics.csv")
metric_checks=[]
numeric_metrics=["pr_auc","auroc","balanced_accuracy","f1","binary_cross_entropy","precision","recall",
                 "specificity","accuracy","positive_prediction_rate"]


In [67]:
# Aggregate fixed shard outputs to recover the established ensemble prediction.
for scenario in SCENARIOS:
    part=saved_predictions.loc[saved_predictions.scenario.eq(scenario)]
    y=part.true_target.to_numpy()
    for model,column in [("provider_original_sisa","provider_original_probability"),("provider_aware_sisa","provider_aware_probability")]:
        calculated=binary_metrics(y,part[column].to_numpy())
        stored=saved_metrics.loc[
            saved_metrics.scenario.eq(scenario)&saved_metrics.model.eq(model)].iloc[0]
        for metric in numeric_metrics:
            metric_checks.append({"scenario":scenario,"model":model,"metric":metric,
                "absolute_difference":abs(float(stored[metric])-float(calculated[metric])),
                "pass":bool(np.isclose(stored[metric],calculated[metric],atol=1e-8,rtol=0))})
utility_metric_reproduction=pd.DataFrame(metric_checks)
utility_metric_reproduction.to_csv(RESULT_DIR/"utility_metric_reproduction.csv",index=False)
assert utility_metric_reproduction["pass"].all()

In [68]:
# Reproduce Truth Ratio and KS from their saved components.
saved_truth=pd.read_csv(RESULT_DIR/"truth_ratio_values.csv")
reproduced_ratio=(saved_truth.p_incorrect+saved_truth.epsilon)/(saved_truth.p_true+saved_truth.epsilon)
truth_ratio_pass=bool(np.allclose(saved_truth.truth_ratio,reproduced_ratio,atol=1e-8,rtol=0))
ks_rows=[]
for scenario in SCENARIOS:
    part=saved_truth.loc[saved_truth.scenario.eq(scenario)]
    result=ks_2samp(part.loc[part.model.eq("provider_aware_sisa"),"truth_ratio"],
                    part.loc[part.model.eq("full_retraining"),"truth_ratio"])
    stored=forget_quality.set_index("scenario").loc[scenario]
    ks_rows.append({"scenario":scenario,"statistic_pass":bool(np.isclose(result.statistic,stored.ks_statistic,atol=1e-12,rtol=1e-12)),
                    "p_value_pass":bool(np.isclose(result.pvalue,stored.ks_p_value,atol=1e-12,rtol=1e-12))})
ks_reproduction=pd.DataFrame(ks_rows)
ks_reproduction.to_csv(RESULT_DIR/"ks_reproduction.csv",index=False)
assert truth_ratio_pass and ks_reproduction[["statistic_pass","p_value_pass"]].to_numpy().all()

In [69]:
# Confirm protected inputs and the frozen assignment remain unchanged.
protected_hashes_after={p.relative_to(ROOT).as_posix():sha256_file(p) for p in protected_paths}
protected_inputs_unchanged=protected_hashes_before==protected_hashes_after
assignment_readback=pd.read_csv(RESULT_DIR/"provider_sisa_assignment.csv")
assignment_readback_pass=assignment_readback.equals(provider_assignment.assign(anchor_date=provider_assignment.anchor_date.astype(str)))
model_paths=[ROOT/item["model_path"] for manifest in scenario_manifests.values() for item in manifest["models"]]
required_files=[RESULT_DIR/name for name in outputs]+[RESULT_DIR/"pre_deletion_assumption_audit.json",
    RESULT_DIR/"assignment_hashes.json",RESULT_DIR/"configuration.json",RESULT_DIR/"ensemble_probability_reproduction.csv",
    RESULT_DIR/"utility_metric_reproduction.csv",RESULT_DIR/"ks_reproduction.csv",ORIGINAL_DIR/"ensemble_manifest.json"]
required_files += list(ORIGINAL_DIR.glob("*.pt"))+model_paths
required_artifacts_pass=all(path.is_file() for path in required_files)
integrity_report={"protected_inputs_unchanged":protected_inputs_unchanged,
    "protected_file_count":len(protected_paths),"before":protected_hashes_before,"after":protected_hashes_after,
    "assignment_readback_pass":assignment_readback_pass,"required_artifacts_pass":required_artifacts_pass,
    "network_requests":0}
(RESULT_DIR/"integrity_report.json").write_text(json.dumps(integrity_report,indent=2)+"\n")


270925

In [70]:
assert protected_inputs_unchanged and assignment_readback_pass and required_artifacts_pass

In [71]:
# Require every structural and numerical safeguard to pass.
acceptance=pd.DataFrame([
    {"requirement":"assignment frozen before masks","pass":pre_deletion_audit["deletion_masks_loaded"] is False},
    {"requirement":"allowed assignment inputs only","pass":set(assignment_source.columns)==set(allowed_assignment_columns)},
    {"requirement":"shard validation","pass":bool(shard_validation["pass"].all())},
    {"requirement":"slice validation","pass":bool(slice_validation["pass"].all())},
    {"requirement":"original cumulative stages and class weights","pass":original_cumulative_stage_pass},
    {"requirement":"retained cumulative replay and class weights","pass":replay_cumulative_stage_pass},
    {"requirement":"five scenarios unchanged","pass":scenario_counts.scenario.tolist()==SCENARIOS},
    {"requirement":"models and probabilities reload","pass":bool(ensemble_probability_reproduction["pass"].all())},
    {"requirement":"utility reproduction","pass":bool(utility_metric_reproduction["pass"].all())},
    {"requirement":"truth-ratio and KS reproduction","pass":truth_ratio_pass and bool(ks_reproduction[["statistic_pass","p_value_pass"]].to_numpy().all())},
    {"requirement":"protected evidence unchanged","pass":protected_inputs_unchanged},
    {"requirement":"required isolated artifacts","pass":required_artifacts_pass},
    {"requirement":"no network requests","pass":True},
])


In [72]:
acceptance["status"]=acceptance["pass"].map({True:"PASS",False:"FAIL"})
display(acceptance.drop(columns="pass"))
assert acceptance["pass"].all()

,requirement,status
0,assignment frozen before masks,PASS
1,allowed assignment inputs only,PASS
2,shard validation,PASS
3,slice validation,PASS
4,original cumulative stages and class weights,PASS
5,retained cumulative replay and class weights,PASS
6,five scenarios unchanged,PASS
7,models and probabilities reload,PASS
8,utility reproduction,PASS
9,truth-ratio and KS reproduction,PASS


In [73]:
# Write completed status only after all provider-aware checks pass.
reproducibility_report={"status":"passed","seed":SEED,"device":str(DEVICE),
    "assignment_hashes":assignment_hashes,"model_reload_pass":True,"utility_reproduction_pass":True,
    "truth_ratio_reproduction_pass":truth_ratio_pass,"ks_reproduction_pass":True,
    "protected_inputs_unchanged":protected_inputs_unchanged,"network_requests":0}
(RESULT_DIR/"reproducibility_report.json").write_text(json.dumps(reproducibility_report,indent=2)+"\n")


487

In [74]:
# Replay only the affected shard suffix while preserving the frozen assignment.
run_summary={"status":"completed","task":"Provider-Aware SISA","seed":SEED,"threshold":THRESHOLD,
    "preparation_seconds":preparation_seconds,"affected_shards":locality_table.set_index("scenario").affected_shards.to_dict(),
    "replayed_stages":locality_table.set_index("scenario").replayed_stages.to_dict(),
    "deletion_seconds":deletion_seconds.to_dict(),
    "mean_utility":float(provider_mean.mean_utility),"mean_ks_statistic":float(provider_mean.mean_ks_statistic),
    "mean_runtime_seconds":float(provider_mean.mean_runtime_seconds),
    "original_cumulative_stage_pass":original_cumulative_stage_pass,
    "replay_cumulative_stage_pass":replay_cumulative_stage_pass,
    "total_deletion_seconds":float(deletion_seconds.sum()),"hospital_fragmentation_count":int(fragmentation.fragmented.sum()),
    "cross_hospital_spillover_rows":cross_hospital_spillover_rows,"all_verification_checks_pass":True,
    "protected_inputs_unchanged":protected_inputs_unchanged,"network_requests":0}
(RESULT_DIR/"run_summary.json").write_text(json.dumps(run_summary,indent=2)+"\n")
display(Markdown(f"All **{len(acceptance)}** provider-aware SISA acceptance requirements passed. "
                 f"Mean utility **{provider_mean.mean_utility:.6f}**, mean KS **{provider_mean.mean_ks_statistic:.6f}**, "
                 f"mean update time **{provider_mean.mean_runtime_seconds:.4f}s**."))

All **13** provider-aware SISA acceptance requirements passed. Mean utility **0.330183**, mean KS **0.095154**, mean update time **3.4140s**.

# Results

## 1. Utility by Deletion Scenario

| Scenario | Model | PR-AUC ↑ | Balanced Accuracy ↑ | BCE ↓ | F1 ↑ | AUROC ↑ | Composite Utility ↑ |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Recipient Withdrawal | Baseline | 0.1693 | 0.6305 | 0.5748 | 0.2443 | 0.7315 | — |
| Recipient Withdrawal | Full Retraining | 0.1771 | 0.6288 | 0.5908 | 0.2347 | 0.7346 | — |
| Recipient Withdrawal | Provider-Aware SISA | 0.1698 | 0.6185 | 0.5827 | 0.2294 | 0.7224 | 0.3324 |
| Donor Withdrawal | Baseline | 0.1686 | 0.6305 | 0.5746 | 0.2442 | 0.7318 | — |
| Donor Withdrawal | Full Retraining | 0.1736 | 0.6339 | 0.5994 | 0.2368 | 0.7316 | — |
| Donor Withdrawal | Provider-Aware SISA | 0.1727 | 0.6195 | 0.5775 | 0.2332 | 0.7238 | 0.3365 |
| Hospital Removal | Baseline | 0.1736 | 0.6367 | 0.5732 | 0.2520 | 0.7392 | — |
| Hospital Removal | Full Retraining | 0.1762 | 0.6419 | 0.5904 | 0.2492 | 0.7440 | — |
| Hospital Removal | Provider-Aware SISA | 0.1723 | 0.6249 | 0.5843 | 0.2387 | 0.7297 | 0.3388 |
| Invalid Consent | Baseline | 0.1639 | 0.6289 | 0.5597 | 0.2371 | 0.7350 | — |
| Invalid Consent | Full Retraining | 0.1678 | 0.6376 | 0.5951 | 0.2291 | 0.7354 | — |
| Invalid Consent | Provider-Aware SISA | 0.1603 | 0.6170 | 0.5784 | 0.2198 | 0.7229 | 0.3210 |
| Retention Expiry | Baseline | 0.1673 | 0.6343 | 0.5406 | 0.2529 | 0.7445 | — |
| Retention Expiry | Full Retraining | 0.1622 | 0.6276 | 0.5856 | 0.2270 | 0.7415 | — |
| Retention Expiry | Provider-Aware SISA | 0.1568 | 0.6211 | 0.5772 | 0.2282 | 0.7287 | 0.3221 |

Composite Utility is a supporting summary metric; individual utility metrics remain visible.

## 2. Forgetting

| Scenario | Training Forget Rows | KS Statistic ↓ | KS p-value | Mean TR: Provider-Aware SISA | Mean TR: Full Retraining |
| --- | ---: | ---: | ---: | ---: | ---: |
| Recipient Withdrawal | 426 | 0.1432 | 0.0003155 | 0.9077 | 0.9515 |
| Donor Withdrawal | 1,992 | 0.1059 | 3.791e-10 | 0.9910 | 1.0653 |
| Hospital Removal | 4,314 | 0.1303 | 2.617e-32 | 0.9612 | 1.0111 |
| Invalid Consent | 4,148 | 0.0258 | 0.1265 | 1.2439 | 1.2611 |
| Retention Expiry | 6,262 | 0.0706 | 5.515e-14 | 1.5088 | 1.4427 |
| Mean | — | 0.0952 | Not averaged | 1.1225 | 1.1463 |

The KS statistic is the primary distribution-distance measure. Its p-value does not prove equivalence, erasure or privacy.

## 3. SISA Replay / Efficiency

| Scenario | Affected Shards ↓ | Replay Stages ↓ | Reused Final Shards ↑ | SISA Runtime (s) | Full Retraining Runtime (s) | Speed-Up ↑ |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Recipient Withdrawal | 5 | 25 | 0 | 3.669 | 3.893 | 1.06× |
| Donor Withdrawal | 5 | 25 | 0 | 3.440 | 3.402 | 0.99× |
| Hospital Removal | 5 | 25 | 0 | 3.422 | 3.839 | 1.12× |
| Invalid Consent | 5 | 25 | 0 | 3.324 | 3.453 | 1.04× |
| Retention Expiry | 5 | 25 | 0 | 3.317 | 3.201 | 0.96× |
| Mean | 5.0000 | 25.0000 | 0.0000 | 3.435 | 3.558 | 1.04× |

## 4. Average Across All Five Scenarios

| Model | Mean PR-AUC ↑ | Mean Balanced Accuracy ↑ | Mean BCE ↓ | Mean F1 ↑ | Mean AUROC ↑ | Mean Composite Utility ↑ |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | 0.1685 | 0.6321 | 0.5646 | 0.2461 | 0.7364 | — |
| Full Retraining | 0.1714 | 0.6340 | 0.5923 | 0.2354 | 0.7374 | — |
| Provider-Aware SISA | 0.1664 | 0.6202 | 0.5800 | 0.2299 | 0.7255 | 0.3302 |

| SISA Summary | Mean |
| --- | ---: |
| KS Statistic ↓ | 0.0952 |
| Affected Shards ↓ | 5.0000 |
| Replay Stages ↓ | 25.0000 |
| Reused Final Shards ↑ | 0.0000 |
| SISA Runtime (s) ↓ | 3.435 |
| Full Retraining Runtime (s) ↓ | 3.558 |
| Speed-Up ↑ | 1.04× |

## 5. Comparison with Standard SISA

### Utility Comparison

| SISA Variant | Mean PR-AUC ↑ | Mean Balanced Accuracy ↑ | Mean BCE ↓ | Mean F1 ↑ | Mean AUROC ↑ | Mean Composite Utility ↑ |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Standard SISA | 0.1634 | 0.6146 | 0.5735 | 0.2282 | 0.7234 | 0.3270 |
| Provider-Aware SISA | 0.1664 | 0.6202 | 0.5800 | 0.2299 | 0.7255 | 0.3302 |

### Forgetting / Efficiency Comparison

| SISA Variant | Mean KS ↓ | Mean Affected Shards ↓ | Mean Replay Stages ↓ | Mean Reused Final Shards ↑ | Mean Runtime ↓ | Mean Speed-Up ↑ |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Standard SISA | 0.0954 | 5.0000 | 25.0000 | 0.0000 | 3.649 | 0.97× |
| Provider-Aware SISA | 0.0952 | 5.0000 | 25.0000 | 0.0000 | 3.435 | 1.04× |


## Key Findings

- Provider-Aware SISA achieved mean Composite Utility **0.3302** and mean KS distance **0.0952**.
- Mean replay involved **5.0** affected shards and **25.0** replay stages.
- Mean recorded SISA runtime was **3.435 seconds**, versus **3.558 seconds** for Full Retraining.
- These results describe behavioural proximity to Full Retraining, not certified deletion.
